# 20 Bibliographic-Coupling Edge Randomization with Prior-Citation Weighting

## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

BC_DATA_DIR = PROJECT_ROOT / "outputs" / "04_bibliographic_coupling_network" / "data"
COCIT_DATA_DIR = PROJECT_ROOT / "outputs" / "03_cocitation_network" / "data"

NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "20_bib_coupling_edge_randomization_prior_weighted"
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / "figures"
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / "tables"
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "data"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENDER_CATEGORIES = ["MM", "MW", "WM", "WW"]

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Functions

In [2]:
def standardize_edges(edges):
    out = edges[["citing_paper_id", "cited_paper_id"]].copy()
    out["citing_paper_id"] = out["citing_paper_id"].astype(str)
    out["cited_paper_id"] = out["cited_paper_id"].astype(str)
    return out

def load_final_core():
    papers = pd.read_pickle(PROCESSED_DIR / "papers_final.pkl")
    cs_papers = pd.read_pickle(PROCESSED_DIR / "cs_papers_final.pkl")
    all_edges = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_final.pkl"))
    all_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_all_to_cs_final.pkl"))
    cs_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_cs_to_cs_final.pkl"))
    papers["id"] = papers["id"].astype(str)
    cs_papers["id"] = cs_papers["id"].astype(str)
    return papers, cs_papers, all_edges, all_to_cs, cs_to_cs

def known_cs(cs_papers):
    return cs_papers[cs_papers["first_last_cat"].isin(GENDER_CATEGORIES)].copy()

def aggregate_oe(observed, expected, baseline, citation_slice):
    rows = []
    for g in GENDER_CATEGORIES:
        o = float(observed.get(g, 0.0))
        e = float(expected.get(g, 0.0))
        oe = o / e if e > 0 else np.nan
        rows.append({
            "gender_cat": g,
            "observed_citations": o,
            "expected_citations": e,
            "oe_ratio": oe,
            "pct_diff_from_expected": (oe - 1.0) * 100 if pd.notna(oe) else np.nan,
            "baseline": baseline,
            "citation_slice": citation_slice,
        })
    return pd.DataFrame(rows)

def standardize_bc(net):
    out = net[["paper_i", "paper_j", "shared_references"]].rename(
        columns={"shared_references": "weight"}
    ).copy()
    out["paper_i"] = out["paper_i"].astype(str)
    out["paper_j"] = out["paper_j"].astype(str)
    out["weight"] = pd.to_numeric(out["weight"], errors="coerce")
    return out[out["weight"].gt(0) & out["paper_i"].ne(out["paper_j"])].drop_duplicates(["paper_i", "paper_j"])

def standardize_cocit(net):
    out = net[["paper_a", "paper_b", "shared_citers"]].rename(
        columns={"paper_a": "paper_i", "paper_b": "paper_j", "shared_citers": "weight"}
    ).copy()
    out["paper_i"] = out["paper_i"].astype(str)
    out["paper_j"] = out["paper_j"].astype(str)
    out["weight"] = pd.to_numeric(out["weight"], errors="coerce")
    return out[out["weight"].gt(0) & out["paper_i"].ne(out["paper_j"])].drop_duplicates(["paper_i", "paper_j"])

## Build Replacement Neighborhoods

In [3]:
papers, cs_papers, citation_edges_final, edges_all_to_cs, edges_cs_to_cs = load_final_core()

cs_known = known_cs(cs_papers).dropna(subset=["year"]).copy()
cs_known["year"] = cs_known["year"].astype(int)

year_map = papers.set_index("id")["year"]

meta = cs_known[["id", "year", "decade", "first_last_cat"]].rename(
    columns={"id": "paper_id", "first_last_cat": "gender_cat"}
)
meta["paper_id"] = meta["paper_id"].astype(str)

bib = standardize_bc(
    pd.read_pickle(BC_DATA_DIR / "net_bib_coupling_topk.pkl")
)

d = pd.concat([
    bib.rename(columns={"paper_i": "paper_id", "paper_j": "neighbor_id"}),
    bib.rename(columns={"paper_j": "paper_id", "paper_i": "neighbor_id"}),
], ignore_index=True)

focal = meta[["paper_id", "decade"]].rename(
    columns={"decade": "focal_decade"}
)
neigh = meta.rename(
    columns={
        "paper_id": "neighbor_id",
        "decade": "neighbor_decade",
        "year": "neighbor_year",
        "gender_cat": "neighbor_gender",
    }
)

valid = d.merge(focal, on="paper_id", how="inner").merge(
    neigh,
    on="neighbor_id",
    how="inner",
)

valid = valid[
    valid["paper_id"].ne(valid["neighbor_id"])
    & valid["focal_decade"].eq(valid["neighbor_decade"])
    & valid["weight"].gt(0)
].copy()

## Historical Prior-Citation Counts

In [4]:
hist = citation_edges_final.copy()
hist["citing_year"] = hist["citing_paper_id"].map(year_map)
hist = hist.dropna(subset=["citing_year"])
hist["citing_year"] = hist["citing_year"].astype(int)
hist = hist[hist["cited_paper_id"].isin(set(meta["paper_id"]))]

hist_group = (
    hist.groupby(["citing_year", "cited_paper_id"])
    .size()
    .rename("n")
    .reset_index()
)

def edge_groups(edges):
    x = edges[edges["cited_paper_id"].isin(set(meta["paper_id"]))].copy()
    x["citing_year"] = x["citing_paper_id"].map(year_map)
    x = x.dropna(subset=["citing_year"])
    x["citing_year"] = x["citing_year"].astype(int)
    return (
        x.groupby(["citing_year", "cited_paper_id"])
        .size()
        .rename("edge_count")
        .reset_index()
    )

all_groups = edge_groups(edges_all_to_cs)
cs_groups = edge_groups(edges_cs_to_cs)

## Prior-Weighted Randomization

In [5]:
gender_map = meta.set_index("paper_id")["gender_cat"]

def evaluate_groups(groups, citation_slice):
    prior = pd.Series(dtype=float)
    hist_by_year = {
        int(y): g.set_index("cited_paper_id")["n"]
        for y, g in hist_group.groupby("citing_year")
    }
    hist_years = sorted(hist_by_year)
    pointer = 0
    pieces = []

    for y, g in groups.groupby("citing_year", sort=True):
        y = int(y)

        while pointer < len(hist_years) and hist_years[pointer] < y:
            add = hist_by_year[hist_years[pointer]]
            prior = prior.add(add, fill_value=0)
            pointer += 1

        targets = set(g["cited_paper_id"])
        n = valid[valid["paper_id"].isin(targets)].copy()

        if n.empty:
            continue

        n["prior"] = n["neighbor_id"].map(prior).fillna(0)

        for variant in [
            "network_weight_only",
            "network_x_prior_citation",
        ]:
            if variant == "network_weight_only":
                n["rw"] = n["weight"]
            else:
                n["rw"] = n["weight"] * (n["prior"] + 1.0)

            wg = (
                n.groupby(["paper_id", "neighbor_gender"])["rw"]
                .sum()
                .unstack(fill_value=0)
                .reindex(columns=GENDER_CATEGORIES, fill_value=0)
            )

            probs = wg.div(wg.sum(axis=1), axis=0).reset_index()

            z = g.merge(
                probs,
                left_on="cited_paper_id",
                right_on="paper_id",
                how="inner",
            )

            for gg in GENDER_CATEGORIES:
                z[gg] = z[gg] * z["edge_count"]

            exp = z[GENDER_CATEGORIES].sum()

            obs = (
                z.assign(
                    cited_gender=z["cited_paper_id"].map(gender_map)
                )
                .groupby("cited_gender")["edge_count"]
                .sum()
                .reindex(GENDER_CATEGORIES, fill_value=0)
            )

            out = aggregate_oe(
                obs,
                exp,
                f"BC_{variant}",
                citation_slice,
            )

            out["citing_year"] = y
            out["replaceable_edges"] = z["edge_count"].sum()
            pieces.append(out)

    yearly = pd.concat(pieces, ignore_index=True)

    agg = (
        yearly.groupby(
            ["baseline", "citation_slice", "gender_cat"],
            as_index=False,
        )
        .agg(
            observed_citations=("observed_citations", "sum"),
            expected_citations=("expected_citations", "sum"),
            replaceable_edges=("replaceable_edges", "sum"),
        )
    )

    agg["oe_ratio"] = (
        agg["observed_citations"]
        / agg["expected_citations"].replace(0, np.nan)
    )
    agg["pct_diff_from_expected"] = (
        agg["oe_ratio"] - 1
    ) * 100

    return agg, yearly

agg_all, year_all = evaluate_groups(all_groups, "All→CS")
agg_cs, year_cs = evaluate_groups(cs_groups, "CS→CS")

results = pd.concat([agg_all, agg_cs], ignore_index=True)
display(results)

,baseline,citation_slice,gender_cat,observed_citations,expected_citations,replaceable_edges,oe_ratio,pct_diff_from_expected
0,BC_network_weight_only,All→CS,MM,"3,027,736.0000","2,935,981.6112",3801614,1.0313,3.1252
1,BC_network_weight_only,All→CS,MW,"296,148.0000","319,565.5837",3801614,0.9267,-7.3279
2,BC_network_weight_only,All→CS,WM,"375,146.0000","422,807.1196",3801614,0.8873,-11.2725
3,BC_network_weight_only,All→CS,WW,"102,584.0000","123,259.6855",3801614,0.8323,-16.7741
4,BC_network_x_prior_citation,All→CS,MM,"3,027,736.0000","2,984,828.1903",3801614,1.0144,1.4375
5,BC_network_x_prior_citation,All→CS,MW,"296,148.0000","312,867.8231",3801614,0.9466,-5.3441
6,BC_network_x_prior_citation,All→CS,WM,"375,146.0000","392,075.4548",3801614,0.9568,-4.3179
7,BC_network_x_prior_citation,All→CS,WW,"102,584.0000","111,842.5318",3801614,0.9172,-8.2782
8,BC_network_weight_only,CS→CS,MM,"2,531,628.0000","2,459,723.4593",3192648,1.0292,2.9233
9,BC_network_weight_only,CS→CS,MW,"253,925.0000","271,999.7427",3192648,0.9335,-6.6451


## Save Results

In [6]:
results.to_csv(
    TABLE_DIR / "bib_coupling_prior_weighted_randomization_oe_by_gender.csv",
    index=False,
)
pd.concat([year_all, year_cs], ignore_index=True).to_csv(
    TABLE_DIR / "bib_coupling_prior_weighted_randomization_by_year.csv",
    index=False,
)